https://www.researchgate.net/publication/378110832_The_Seasonality_of_Bitcoin

In [ ]:
import holidays
import polars as pl
from zh import (
    ET,
    OPEN_TIME,
    SHARPE,
    scan,
)

data = scan("1m")
nyse_holidays = holidays.financial_holidays(
    "XNYS", years=[2020, 2021, 2022, 2023, 2024, 2025]
)
data_us: pl.LazyFrame = data.with_columns(pl.col("open_time").dt.convert_time_zone(ET))

In [ ]:
n = 60

data_us.filter(
    (pl.col("symbol") == "SOLUSDT")
    & pl.col("open_time").is_between(
        pl.datetime(2023, 1, 1, time_zone=ET),
        pl.datetime(2024, 12, 31, time_zone=ET),
    )
).gather_every(n, offset=n - 1).with_columns(
    pl.date(2020, 1, 1).dt.combine(pl.col("open_time").dt.time()).alias("time"),
    pl.col("close").pct_change().alias("returns"),
).filter(
    pl.col("open_time")
    .dt.weekday()
    .is_in([1, 2, 3, 4, 5])
    .and_(pl.col("open_time").dt.date().is_in(nyse_holidays).not_())
).group_by("time").agg(pl.col("returns").mean()).sort("time").collect().plot.bar(
    x="time", y="returns:Q"
)

In [ ]:
def backtest_single_asset(
    data: pl.LazyFrame | pl.DataFrame,
    symbol: str,
    start_date,
    end_date,
    fee_bps: float = 0.0,
    long=[],
    short=[],
    days=[1, 2, 3, 4, 5, 6, 7],
) -> pl.LazyFrame:
    """
    Backtest a mean reversion strategy on a single asset with 1-minute hold period.

    Parameters:
    -----------
    data : polars.LazyFrame
        close data with columns: symbol, open_time, close
    symbol : str
        Symbol to trade (e.g., 'ETHUSDT')
    start_date : datetime
        Start date for backtest
    end_date : datetime
        End date for backtest
    entry_z : float
        Z-score threshold for entry (default: 2.0)
    fee_bps : float
        Fee in basis points (default: 0)
    window_size : int
        Rolling window size for calculations (default: 14)
    """
    fee_rate = fee_bps / 1e4

    # Prepare data
    asset_data = (
        data.filter(pl.col("symbol") == symbol)
        .select(
            "open_time",
            "open",
            "close",
        )
        .filter((pl.col("open_time") >= start_date) & (pl.col("open_time") <= end_date))
        .sort("open_time")
    )

    # Backtest
    bt = (
        asset_data.with_columns(
            (pl.col("close").log().diff()).alias("returns"),
        )
        .filter(pl.col("returns").is_not_null())
        .with_columns(
            pl.when(
                pl.col("open_time").dt.hour().is_in(long)
                & pl.col("open_time").dt.weekday().is_in(days)
                & ~pl.col("open_time").dt.date().is_in(nyse_holidays)
            )
            .then(1)
            .otherwise(
                pl.when(
                    pl.col("open_time").dt.hour().is_in(short)
                    & pl.col("open_time").dt.weekday().is_in(days)
                    & ~pl.col("open_time").dt.date().is_in(nyse_holidays)
                )
                .then(-1)
                .otherwise(0)
            )
            .alias("pos")
        )
        .with_columns(
            pl.col("pos").shift(1).fill_null(0).alias("pos_prev"),
        )
        .with_columns(
            (pl.col("pos") * pl.col("returns")).alias("pnl_gross"),
            (pl.col("pos") - pl.col("pos_prev")).abs().alias("turnover"),
        )
        .with_columns((fee_rate * pl.col("turnover")).alias("fees"))
        .with_columns((pl.col("pnl_gross") - pl.col("fees")).alias("ret"))
        .fill_null(0)
    )

    return bt.lazy()

In [ ]:
backtest_single_asset(
    data_us,
    symbol="DOGEUSDT",
    start_date=pl.datetime(2020, 1, 1, time_zone=ET),
    end_date=pl.datetime(2025, 12, 31, time_zone=ET),
    fee_bps=5,
    short=[9, 10, 11],
    long=[17, 18],
    days=[1, 2, 3, 4, 5],
).with_columns(
    pl.col("ret").cum_sum(), OPEN_TIME.dt.convert_time_zone("UTC")
).group_by_dynamic(pl.col("open_time"), every="1d").agg(
    pl.col("ret").last()
).collect().plot.line(x="open_time", y="ret").show()

In [ ]:
backtest_single_asset(
    data_us,
    symbol="DOGEUSDT",
    start_date=pl.datetime(2025, 1, 1, time_zone=ET),
    end_date=pl.datetime(2025, 12, 31, time_zone=ET),
    fee_bps=5,
    short=[9, 10, 11],
    long=[17, 18],
    days=[1, 2, 3, 4, 5],
).select(sharpe=SHARPE).collect()